# Resume the interrupted Stage 2 test run

**Recovery revision: gpu-split-v2.**

Start a **fresh Kaggle notebook session** with **Internet** and **GPU T4 ×2** enabled. Attach your previous **stage2_test_bundle.zip** as an input. The configuration automatically finds that filename under `/kaggle/input` and also handles an automatically unpacked ZIP; if several copies exist, set `RESUME_BUNDLE` to the exact path. Then use **Save Version → Save & Run All** or Run All.

Only `no_contract_to_generator` is selected. The runner restores all six systems, keeps existing task/seed records and skips completed tasks. With the inspected checkpoint, it starts with **562/600 records**, including **62/100** in the unfinished arm, and resumes at **taxi_dedup_003** (38 pending). No retraining or GitHub push is needed.

To address the CUDA OOM, a process-local wrapper explicitly places the 7B four-bit repair model on GPU 0 and the 3B FP16 generator on GPU 1. The frozen source checkout, prompts, parser inputs, precision, decoding and budgets are unchanged. The bundle records the placement intervention, affected session and actual model devices. GPU placement can affect numerical results; this recovery is not claimed to be bitwise equivalent to the earlier automatic placement. The GPU run itself still needs validation.

Download the new **stage2_test_bundle.zip** when finished. Success requires `experiment_complete: true` and **600/600 records**.


In [1]:
# Frozen configuration. Select fewer systems only to split the same experiment across sessions.
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"
BRANCH = "extra"
REVISION = "e9aaa31b85225fc14792c3d34b9f24de149f92b1"
GENERATOR_PROMPT = "b2-v2"
AGENT_PROMPT = "a1-v4"
EXPECTED_GENERATOR_HASH = "ed85f90bc61a"
EXPECTED_AGENT_HASH = "af55b596106e"
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"
GENERATOR_4BIT = False
AGENT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
AGENT_4BIT = True
B3_PARSES = "kaggle_outputs/parsing/7b_p1-v3/parses.jsonl"
FT_PARSES = "kaggle_outputs/parsing/ft_1_5b/parses.jsonl"
FT_TAG = "ft_1_5b"
STAGE = "test"
BACKEND = "hf"
SEEDS = [0]
TEMPERATURE = 0.0
MAX_ATTEMPTS = 3
MAX_TOOL_CALLS = 8
TOKEN_BUDGET = 60000
GENERATOR_MAX_NEW_TOKENS = 512
DTYPE = "float16"
EXPECTED_INFERENCE = {"torch": "2.11.0+cu130", "transformers": "5.20.2",
                      "bitsandbytes": "0.50.2", "peft": "0.21.2"}
ALL_SYSTEMS = ["baseline1", "baseline2_repair", "baseline3_repair", "gold_contract",
               "proposed", "no_contract_to_generator"]
SYSTEMS = ["no_contract_to_generator"]  # Resume this arm; require the other five to be complete
RESUME_BUNDLE = "auto"       # optional: "/kaggle/input/my-stage2-results/stage2_test_bundle.zip"
WORK = "/kaggle/working"
assert STAGE == "test" and BACKEND == "hf" and SEEDS == [0] and TEMPERATURE == 0.0
assert GENERATOR_PROMPT == "b2-v2" and AGENT_PROMPT == "a1-v4"
assert (GENERATOR_MODEL, GENERATOR_4BIT, AGENT_MODEL, AGENT_4BIT) == ("Qwen/Qwen2.5-3B-Instruct", False, "Qwen/Qwen2.5-7B-Instruct", True)
assert (MAX_ATTEMPTS, MAX_TOOL_CALLS, TOKEN_BUDGET, GENERATOR_MAX_NEW_TOKENS, DTYPE) == (3, 8, 60000, 512, "float16")
assert SYSTEMS and len(set(SYSTEMS)) == len(SYSTEMS) and set(SYSTEMS) <= set(ALL_SYSTEMS)
print("Selected systems:", SYSTEMS)

Selected systems: ['no_contract_to_generator']


In [2]:
# Fetch the exact development source commit. Authentication is optional for the public repo.
import os, sys, json, subprocess, hashlib, time, zipfile, stat
from pathlib import Path, PurePosixPath
REPO = Path(WORK) / "repo_stage2_test_frozen"
if not REPO.exists():
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        clone_url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    except Exception:
        pass
    result = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, clone_url, str(REPO)],
                            capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr.replace(clone_url, REPO_URL))
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip(), "Use a clean source checkout"
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if commit != REVISION:
    subprocess.run(["git", "fetch", "origin", REVISION], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
os.chdir(REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert COMMIT == REVISION
print("Frozen source commit:", COMMIT)

Frozen source commit: e9aaa31b85225fc14792c3d34b9f24de149f92b1


In [3]:
# Frozen development source/data plus both parser files. Only hashes are embedded, not answers.
FROZEN_SOURCES = {
  "benchmark/prompts/schema_b1-v1.json": "6fe84f85b3b50de0dd8303a4974da1154ff4400ed3b1daadb0fa698e28d57852",
  "benchmark/tasks/adult/adult_col_001.yaml": "670f19fee6c3f8497f0518b637d70c3a877eaccfa5b4da8a5a642c787e60001d",
  "benchmark/tasks/adult/adult_col_002.yaml": "b51dfe659a8761d8011a56c126e8e391cd4bea97c75213ee503b09b856bd9725",
  "benchmark/tasks/adult/adult_col_003.yaml": "75e14a1535f201b4f762edc299c5f1d32db1e4f1c7b4ce8d48e785e636790901",
  "benchmark/tasks/adult/adult_col_004.yaml": "557b5d16d143cb9698fe3817bc0c7c5d61e0a86b5d9cef8121630e0e03c06fa7",
  "benchmark/tasks/adult/adult_col_005.yaml": "24d7ab881ed7f0b616f351f86b724d2726008c67af564c02e0572390863192c4",
  "benchmark/tasks/adult/adult_col_006.yaml": "51a83f08442e6b00e471f13d2dc10d3671e55737c94c49c0593a28e79950d41c",
  "benchmark/tasks/adult/adult_conv_001.yaml": "4f0e21e69de69050eeb0cb036b3ff2fcaf3642c35aa7f7bbcf36e8258f405ce2",
  "benchmark/tasks/adult/adult_conv_002.yaml": "d5e9d26bb62740ae74553f648c7e6593f532535fbc7cf59a7fbd459937e97621",
  "benchmark/tasks/adult/adult_conv_003.yaml": "0f1a3c4ccacd397e9457647f43bd1f2c8a8239a62c0eb653b0abd11128ea535d",
  "benchmark/tasks/adult/adult_conv_004.yaml": "d468e742eec10b91291dcd9ae128dacb0139bbcda50100e50ee7b7679f852fb1",
  "benchmark/tasks/adult/adult_conv_005.yaml": "5abea53f8559212e969b61aae10a2aec31196daf737ec42a09acedee24794da5",
  "benchmark/tasks/adult/adult_conv_006.yaml": "532824aa58a7399f806c355f49fbd77907cb57ef285d7e57fa43d955ebcb9a7b",
  "benchmark/tasks/adult/adult_dedup_001.yaml": "1f84536d6430967d77674e1d639154623fdad67cf2050aca25bfabd458fded77",
  "benchmark/tasks/adult/adult_dedup_002.yaml": "70eb5b3ab9e23d84ba41de4e8dcd93a4924e57000df2a7fcfb6f43fc0889fc03",
  "benchmark/tasks/adult/adult_dedup_003.yaml": "46fce6c6312a6c4f39eca78aab5f0e40c318b0664f7975a64a9715900b55dc0a",
  "benchmark/tasks/adult/adult_dedup_004.yaml": "c423df20aa8e24300868d3432051ab4be320999f750155cdcd007d8e495368d3",
  "benchmark/tasks/adult/adult_dedup_005.yaml": "6b092fe35193023eb4b3124e360c843836e3157ee509875ba9efb8aaafbf011b",
  "benchmark/tasks/adult/adult_dedup_006.yaml": "88dddf811e4e7b971da0df7405b2d74fa0692c30fc7b10091941411f8be8f5e7",
  "benchmark/tasks/adult/adult_filt_001.yaml": "b5eee0f7093ae184ad3b37b8a9ada4e6a11e91f5c3544636edaaea35eee1632f",
  "benchmark/tasks/adult/adult_filt_002.yaml": "4d49ca67ab95a0114921806fa1a761cf1e8eaaf58efd8d7c786602f8a117b125",
  "benchmark/tasks/adult/adult_filt_003.yaml": "cecc8ce546565e46f1635cf2783d13e920ed7b63c31ffc45fe47c974a45e54e5",
  "benchmark/tasks/adult/adult_filt_004.yaml": "0c4372394b01c94b95d6a23a862b4520eb037eef67ad179d03dbde989ff8457e",
  "benchmark/tasks/adult/adult_filt_005.yaml": "f86b7bcd5e9838fc71c3665d82bb9a34432755aab1e7f9a24715a39e114e0fa1",
  "benchmark/tasks/adult/adult_filt_006.yaml": "667aa4eea5fde93a698dedec0df0bc8120d67fd609c602534b352856d939477d",
  "benchmark/tasks/adult/adult_miss_001.yaml": "4ef3bc73198f7a33fcb0ab1e35cd64ebd87b5dd59e6b41ef05ac0e9ddd937dfc",
  "benchmark/tasks/adult/adult_miss_002.yaml": "1311e837893c68d33a56c8e5ae96e22eb74554c7b768d3285f1ea7e60497722b",
  "benchmark/tasks/adult/adult_miss_003.yaml": "ef49d67ddf33f110b9ccd3211375abc2a5cd21ec9a036874499f0f34e15536d5",
  "benchmark/tasks/adult/adult_miss_004.yaml": "1f7c3e4cd24d6af89ecd6a2122d7ec26d53e510ec81e001560bde99b0c40ae33",
  "benchmark/tasks/adult/adult_miss_005.yaml": "ea69da6b50bf309958ee06aa09dcf92d126f35e2906816decba3e5bd0f9e390a",
  "benchmark/tasks/adult/adult_miss_006.yaml": "ffebdcc54e3328dd76c445bbafada91b963952398eb7c013ae951b24c211178b",
  "benchmark/tasks/airbnb/airbnb_col_001.yaml": "c56754dcc3354ee828cce3c31495f683a0bbb45a2782d08bd8b70cb056345ca4",
  "benchmark/tasks/airbnb/airbnb_col_002.yaml": "387152c68631b099b4876e46956fe72f3497fc15772f01612e810f62ddb6e55d",
  "benchmark/tasks/airbnb/airbnb_col_003.yaml": "baedb0eadf1ccaf10f5d0776ae6077b5e152df3b4c86b0df6dee1ccac54c6465",
  "benchmark/tasks/airbnb/airbnb_col_004.yaml": "82e3f61098c8c6094b47f346f49a3a6c5ed9b03f0e79104d0a87b3a66d1f31d7",
  "benchmark/tasks/airbnb/airbnb_col_005.yaml": "f0a96d8090ac54057b7eca4225c1d23fce4beff1c2eec61dfe506dc7724ab7e2",
  "benchmark/tasks/airbnb/airbnb_col_006.yaml": "84edba8a5acef808785c805ee644cd2e5c8837c931d13164c1f71efc1ba1757a",
  "benchmark/tasks/airbnb/airbnb_conv_001.yaml": "f4ba190b7ca55e5ca16af69bd381d047b2e8a0ec927bdf207b3600399f0b1601",
  "benchmark/tasks/airbnb/airbnb_conv_002.yaml": "17e4e0f979dcb27c4cb8a31e85f12fad4cf31cdead300565426b945c7d21f1c9",
  "benchmark/tasks/airbnb/airbnb_conv_003.yaml": "e62e9c1af9b92cfe525f7bc55d229f64bc03282487e78e07a88f58b24b0368ee",
  "benchmark/tasks/airbnb/airbnb_conv_004.yaml": "3afaa8a30f44a761ce06fd36dd8d815fb8d596b5a762d6cf8913ea9983671f38",
  "benchmark/tasks/airbnb/airbnb_conv_005.yaml": "9eeb49b9e52f9f6521a9d777c269e6d4574abf4848781e15fa2ea4599a836105",
  "benchmark/tasks/airbnb/airbnb_conv_006.yaml": "7f3857d908e34a9735c05b307b688d4750238788b4df206dd8a814e16d382428",
  "benchmark/tasks/airbnb/airbnb_dedup_001.yaml": "519ed3e4d6e79f3938b6b3dc6dc869500be30aa1b391bff7a7eb107cef997767",
  "benchmark/tasks/airbnb/airbnb_dedup_002.yaml": "c62728c17b0a104525ce85da4d2c48d565b027026e3807d2140e42041bd73b39",
  "benchmark/tasks/airbnb/airbnb_dedup_003.yaml": "aa41903d53696f75c4cabda7202af8b0f3ce729c751e84ad84ba7b8e067f48e2",
  "benchmark/tasks/airbnb/airbnb_dedup_004.yaml": "37e23e982d173f7e5b6a639c553532c0aa028e8959dff6e8131ce8f32f621bde",
  "benchmark/tasks/airbnb/airbnb_dedup_005.yaml": "16a9d054d32e3bca55ac4ae0b57381e11eb0f2c5d162bb4dca7fd18a75b0c6a8",
  "benchmark/tasks/airbnb/airbnb_dedup_006.yaml": "c7f59d0826400daf52ecc049f2399a4a4e1b17cb8e7c80072978e358f689ea7a",
  "benchmark/tasks/airbnb/airbnb_filt_001.yaml": "f730cb0980091e309c395496014d1c1602f65bd949754d3b248af13f8fcf6b5b",
  "benchmark/tasks/airbnb/airbnb_filt_002.yaml": "22b8655ec7bb2178f363cd36f1c665033539bf28c7ee4b341219587e67b90853",
  "benchmark/tasks/airbnb/airbnb_filt_003.yaml": "89908fdd13523ccd184bfd76e92c2c4f27f73609068023ec2807ced342749827",
  "benchmark/tasks/airbnb/airbnb_filt_004.yaml": "1b8b80cf899a25325229d14dee8cc742c0deaccecb47d686567c171770851595",
  "benchmark/tasks/airbnb/airbnb_filt_005.yaml": "8580a5daf7d34107765263f2820e54a6911a95b8792f854e6086ad911dd3d296",
  "benchmark/tasks/airbnb/airbnb_filt_006.yaml": "1e5673ba0646308e9685d8cd66f5d41221a8c851419f038fda4c221483f0de6e",
  "benchmark/tasks/airbnb/airbnb_miss_001.yaml": "0db5d30e7e00b7d853bd726dd07889f64280524b6d99d34318bb18fb8beb7809",
  "benchmark/tasks/airbnb/airbnb_miss_002.yaml": "9021b50c4d481413751fdb647ac63e66994eeec5e9d507da32dd105193c659da",
  "benchmark/tasks/airbnb/airbnb_miss_003.yaml": "6c14263a25f916ca53c49d5a18751af63655855fb9566e48d0bb174b64d582f7",
  "benchmark/tasks/airbnb/airbnb_miss_004.yaml": "1b0b25a576730c16eae10f775e965ace4685cb594bd8573201a08d668d31edf9",
  "benchmark/tasks/airbnb/airbnb_miss_005.yaml": "6e4cf4025e6298372745c8bbb4b8f6ffd8be94860f933a30e8c0dd0def84d80f",
  "benchmark/tasks/airbnb/airbnb_miss_006.yaml": "3385f3a5e535733547813df625d3811f90cf99e70c12258913b95df47683fbdc",
  "benchmark/tasks/taxi/taxi_col_001.yaml": "54cf851778066a0ad0b23b5a54fa858e24797a8a107678fd63d3f57ef6d9df81",
  "benchmark/tasks/taxi/taxi_col_002.yaml": "11599f8b661150bece195ef4c83cdf007714f776b9b13ff352a858851c14e5ad",
  "benchmark/tasks/taxi/taxi_col_003.yaml": "6135579b9831099d62ea67f04a8d896ae219701fcff99f2864c546ea67d5fe3a",
  "benchmark/tasks/taxi/taxi_col_004.yaml": "6e40dbb9b431846ad453f2a619199286ca3c6b12f6b334aa54ac71ff62a9cc30",
  "benchmark/tasks/taxi/taxi_col_005.yaml": "3ab5fa1a49e1c282be840ab99c4057a0d249668bf8f24f0bc8fa35dbfeefaa16",
  "benchmark/tasks/taxi/taxi_col_006.yaml": "5fb71b82c1192ea81e54bd38bd8096b67bdbde7f839b62066e99b9552ed71b52",
  "benchmark/tasks/taxi/taxi_conv_001.yaml": "04288a367a2a2d6c19b148d72f78d6e589b774181db9dcb2496b92ee215bdc1f",
  "benchmark/tasks/taxi/taxi_conv_002.yaml": "3fba4a93b84ee78f5924859c1ae6911b8c0f66026bf549e7b9c24d3a7d944114",
  "benchmark/tasks/taxi/taxi_conv_003.yaml": "901a570c40ec4e66190ad2eb293b0e1c1d979fce604a2be72c69c2dbe9bcd8ce",
  "benchmark/tasks/taxi/taxi_conv_004.yaml": "0ba2de267ebccbef67a3f58e4fe01471191b4d15245c4d188f90b7e85fada435",
  "benchmark/tasks/taxi/taxi_conv_005.yaml": "61cb10400068425d6042c509ca1646f3582a9b5e2fefb99a3bf79866c603e0be",
  "benchmark/tasks/taxi/taxi_conv_006.yaml": "65e8055b7d21d60cdcf53a8d93be9141b883e7f07e4868b206f01537d72f4049",
  "benchmark/tasks/taxi/taxi_dedup_001.yaml": "a5a613caaa3d43436c9d60f4f6f2989b4c1f69c31fd5c7a3d99576de05055d96",
  "benchmark/tasks/taxi/taxi_dedup_002.yaml": "8a622851986c6f1ef129251b5a7b79e02d7ada3f7c2dab2fd45c52e7f9cc0971",
  "benchmark/tasks/taxi/taxi_dedup_003.yaml": "da43caf6dcde9575aa92870ea65539ab0d2d3938638378f07fe413b315214a18",
  "benchmark/tasks/taxi/taxi_dedup_004.yaml": "d6d9e3c17cb55c5db58a31103c1d883139f91e02ea0c952c91268000809715ff",
  "benchmark/tasks/taxi/taxi_dedup_005.yaml": "d0e18a15b6db4ab63e737c50bad9c56d8ad4b7cfe02bba2e27bbfd6bacca2fcc",
  "benchmark/tasks/taxi/taxi_dedup_006.yaml": "ef4f9e216802c62ba552268245957d41555ca261d945d6760447852dea154f20",
  "benchmark/tasks/taxi/taxi_filt_001.yaml": "c1d40175f8da2494c98f81feb6ed94f3926a5678c50d9e9bb57eacbf31b32697",
  "benchmark/tasks/taxi/taxi_filt_002.yaml": "65385fd08566cf40670b23e010eb965af162e054f07b36cc04f72a1c2cda403c",
  "benchmark/tasks/taxi/taxi_filt_003.yaml": "ae3368b78f1df9cebff5c23a0ffabb4af44fff3c5dfa14acde25c499bc47ddc6",
  "benchmark/tasks/taxi/taxi_filt_004.yaml": "d9af2f20332d4e7f5c99127e0b159f8179efceeb4386f55eab1a5e218fe44a9d",
  "benchmark/tasks/taxi/taxi_filt_005.yaml": "04908053986abec710928234180afe312496cef881c6caa2ee7932be4dec0aea",
  "benchmark/tasks/taxi/taxi_filt_006.yaml": "43d6b4c1966e56ecbf4179f0ae11de555e3c602466cf5627a4d34bdea3601eeb",
  "benchmark/tasks/taxi/taxi_miss_001.yaml": "6a08ef11322214f45f6dceaa31a864248a8fd041b7ccee588176950a74f177ab",
  "benchmark/tasks/taxi/taxi_miss_002.yaml": "43f7ac74f41ede3f8d877817d6e833ef4a7c3673fc32a4dd3807bb262798455b",
  "benchmark/tasks/taxi/taxi_miss_003.yaml": "a10267dab28d8577a013649b0671e3e5181c23d29c03a86e38e858dc6facacd2",
  "benchmark/tasks/taxi/taxi_miss_004.yaml": "a1451e2081f8eee66be8bcc089ca782d93a7942d4d6a10a3179138efdddc7b63",
  "benchmark/tasks/taxi/taxi_miss_005.yaml": "88bb3c9a6fd5e609183cb9d1de7a9b43b46bffedbce0575e58f7d38ca368518c",
  "benchmark/tasks/taxi/taxi_miss_006.yaml": "7e873c475f7f876de069d5a40b88dc0dbfc903682238c4bfffb11c215b96487f",
  "benchmark/tasks/titanic/titanic_col_001.yaml": "a96bd1ce4c48daaafe95644421de98e68aa681968a694dbb50aec3c9b6fec0f7",
  "benchmark/tasks/titanic/titanic_col_002.yaml": "742b0072f78054829719dcec619f26705e72784fd6425daaf88a54630ece5122",
  "benchmark/tasks/titanic/titanic_col_003.yaml": "331704967db4c894a3e7905206b44310da5e51bf64bb827b80c547e4846d4ee1",
  "benchmark/tasks/titanic/titanic_col_004.yaml": "0068b36c207eccfe5b580acac9988278cb14e9e323dc05867fcee46ed2a7d9e4",
  "benchmark/tasks/titanic/titanic_col_005.yaml": "b562421d88ca6914a00029d6bfdface6e40f40cba02b944bdeee6cf1d0487bf8",
  "benchmark/tasks/titanic/titanic_col_006.yaml": "4cf89d08d312067fa90f2a43222e4ab7ce7e4d1c8199244e207cffd6b93c8edc",
  "benchmark/tasks/titanic/titanic_conv_001.yaml": "2629f0f2f0076d49dec807a4519e3a96c612b25ccd5eeb2a8d026911dbc82c08",
  "benchmark/tasks/titanic/titanic_conv_002.yaml": "74d49b60f9f18cdadb804034c1d14c774ce8ecfed69a7cbb073bdfe99f25320c",
  "benchmark/tasks/titanic/titanic_conv_003.yaml": "3fd1640a9711acbe6c6912c915506bcdc65a4b559e6a9a645803b4473701045a",
  "benchmark/tasks/titanic/titanic_conv_004.yaml": "f8a2316d61e069cd9c0a55c56e2c9eb251731d84683eda9ca04abea20cd09071",
  "benchmark/tasks/titanic/titanic_conv_005.yaml": "96a5d3749f9d239652d414c0fed2dcadbbe8e2688450a51e38fd93166cd374f2",
  "benchmark/tasks/titanic/titanic_conv_006.yaml": "4ec1adc8acb39e517c934f8c4a5492a67c11fa5686c387e40a9c180b5d34a2b6",
  "benchmark/tasks/titanic/titanic_dedup_001.yaml": "fffea1de0ca38c73babe1fa9e4d256bbe5ebb9530894aae3975954978e93b21e",
  "benchmark/tasks/titanic/titanic_dedup_002.yaml": "148a465986a2c95d293969d04b65fa04eb00f939ee205eefa5a65c3de64d17a9",
  "benchmark/tasks/titanic/titanic_dedup_003.yaml": "f5c1948c43422412b4f419fa4c47e75ca858d3a03826dfa52ce325a14a050a23",
  "benchmark/tasks/titanic/titanic_dedup_004.yaml": "f07211d80fcb72675c8768dce8759c7dcc87911a7e2e52ed458d10aae9dc8b2c",
  "benchmark/tasks/titanic/titanic_dedup_005.yaml": "825dae40d8d599c8f9f5d87eff9783f867c48823b988cd77dc40e5525b49bd55",
  "benchmark/tasks/titanic/titanic_dedup_006.yaml": "1cfd8ef79f9ed08423312d7a13dde36ebab8adf5465f51b5ed26431670923496",
  "benchmark/tasks/titanic/titanic_filt_001.yaml": "f2de0f78b63e4a657e4a2ecb1455091590383b4b53980c37681ba3e51dda3fbc",
  "benchmark/tasks/titanic/titanic_filt_002.yaml": "c90fca5ada213d5db4452a0bb865fbb64a8fd1399167812e9953edd080fe9c8a",
  "benchmark/tasks/titanic/titanic_filt_003.yaml": "dcaf29e272573c0dc57c33ffe0dc3593ada8fb032f57aca58b9b8c0ee0f475c5",
  "benchmark/tasks/titanic/titanic_filt_004.yaml": "c0d8c85971608a570f2a2f4d77e99891a3f322ca60abbc262d426390be104082",
  "benchmark/tasks/titanic/titanic_filt_005.yaml": "2b5dda30ebf829b82f927e1d84c44ea0c87a3855cf03d12f7708a64100652f14",
  "benchmark/tasks/titanic/titanic_filt_006.yaml": "76a659439f858c7e665ab1fc0fb21395d706d00d8f0b85df82ff42d7813b196a",
  "benchmark/tasks/titanic/titanic_miss_001.yaml": "221a4ef0f0deb0609934803b7cb1dd4938ebe26b79eece51fb0e2e220bf3318c",
  "benchmark/tasks/titanic/titanic_miss_002.yaml": "f19d635851682fe1171581f8c5ea906a021f05571b85dd843043fa6b3b433b07",
  "benchmark/tasks/titanic/titanic_miss_003.yaml": "5946a06c8f9114984a3c712f2f090ea472a5a788e822f96445c11856dfd4e46b",
  "benchmark/tasks/titanic/titanic_miss_004.yaml": "767ee943d453b44d60c0f115b54e8255ff05da8f6ac1a290b08c27c94ba38c04",
  "benchmark/tasks/titanic/titanic_miss_005.yaml": "477f3a7bb39922ed3ad2fee3c21306a9dd512cac602dc817f1e5ea24294e55ae",
  "benchmark/tasks/titanic/titanic_miss_006.yaml": "a8ea2a8623de07601c767d326db2d3fc26ff7124581d3f95f10e97b6568e0fb8",
  "data/Titanic-Dataset.csv": "7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f",
  "data/adult_dev.csv": "b2ebdd9b244c7fff5ebe1d6a328cc49f37b7a92aa4c2cb6ad0f2b5671e49d2ef",
  "data/airbnb_dev.csv": "c25f044eb00cd2c88bf05661bf50726c00890f8cb3aecd1a4770ec9f032e68a3",
  "data/checksums.json": "5df7205a322856f4bcd6dcb9fc38fc5269e476a40ffd03ffb5a216e2107b05b5",
  "data/yellow_tripdata_2026-01.csv": "b1ac75514b2adec94444606a9b5ec7b60a24b02530144592d2e64a2fce891ecc",
  "eval/agent_metrics.py": "d79fc3597583557260caa89bafe662690561ac883886791d91ba321d7ac5283c",
  "eval/annotation_agreement.py": "6dedc79804af4e358d494353debca48bc6c6d171a223dd058ef82a4e0eea9438",
  "eval/baseline1_by_dataset.py": "469c5e908614e68c002f6cb64ef1a9157a140701c303c53b3586974de9bd2a4d",
  "eval/baseline2.py": "8e9a4158e4df85467b42a821ad62184f7a770a2c1f85100095e6b225f27f5657",
  "eval/build_agent_scenarios.py": "941aec9b717242879b7017ae9322585fb41d68573d8e3bb8ad22728a6bc30345",
  "eval/build_pool.py": "2fb3689c0fca670569779b62117d0ee1da05f4fbb01010b58fb0f7d226ef1229",
  "eval/contract_metrics.py": "cafb6b1ec4a5970b52462ff39f224988d4e61aca4446ce34f6ff435c2c5879e3",
  "eval/demo.py": "ba8fc1b8cf149e2c7dabaf076007b43b4bd887bb654b2811bb954737869ea4cf",
  "eval/draft_error_kinds.py": "6f105596d72d93c8875b2e0f1d03249777837b62e99f0079b8c596cf22e16173",
  "eval/equivalence.py": "35d56f28b00f44834329884dc4d8455959c3df7f669efd3cfae4359c75abeee0",
  "eval/error_kind_breakdown.py": "a0c64a940fed9c340931144ddbde354fbd1e3396a84d6ea70f7172f84c219e1d",
  "eval/ext_common.py": "a51da2cb6c50320bf5519548e5c289b9e2cbbfe00cf4e3d4be8dced984114098",
  "eval/ext_docker_check.py": "69bb28b61315348844550df07597dc64e75e71345d9e1ecc5e9d8ea7750db47d",
  "eval/ext_precheck.py": "f2b9784be57fe8cbadf94183dde3ffa2bb08a0ab17760debff529f434207f649",
  "eval/ext_reask.py": "3cd9ce2c3527d22c269221c0483c9e47b51e0573329aa0f605a73a781c3f126c",
  "eval/ext_replay_no_progress.py": "f330b8abe92a5f260847f8014a940c2210c083df07ff3582023923f4a0b26470",
  "eval/ext_runner_patch.py": "4908b28321faeb863d1b492aaf4fa9abe5932f6899ab0ecc63d0db75f888905a",
  "eval/ext_score.py": "c1102ce290013d704bc3d2c97ee12c98fd580002f55ee233ebc2c5c75977303b",
  "eval/ext_vllm_compare.py": "f8428ca8604da76cc62ea92f46d74456f222a807b182c0b5ded3f5309a6d55df",
  "eval/generate.py": "d477bd632921f581527031a07119a6eccae794df4a9c99ac40f5ac2b719ffce2",
  "eval/label_candidates.py": "2d72bcaf9c16adaaac085e92541ae3b2f91a6ef456ab99b82fda0250602ca450",
  "eval/make_annotation_sheet.py": "da4b48657f617dec84dd8b38b813bb693f7809c8b1cef6f510ef8eda468f5439",
  "eval/metrics.py": "820baaf68953ef2938f7568e6fce75ee083b9f6baaeb541a83d35d07a84d52ea",
  "eval/parse_contracts.py": "13ea743ba2ab66d75b2dd764146931bf789aec7b32167a457143aecee96d8079",
  "eval/parse_contracts_tools.py": "9a7d1bb51fff82606bba827a1de8cacf353ec158eaa3fdfdd1d5b6f897487c3c",
  "eval/parse_contracts_tools_ft.py": "ac19cd6a844fc1fdcb6026d5eca03619179deb9614eeb63e495fd33903e65041",
  "eval/run_config.py": "cd34c54721cd6b5b0dca6a34e27d9467d177cdaaf28d9a5487957ee01dbd0386",
  "eval/run_intent_oracle.py": "4473adeef210421a34ac184b8927bd345fe3909c567d8006d39a125289be8ba4",
  "eval/run_llm_verifier.py": "69a0406aadacc7edf59d47c984f805ca85f2aad800adf46019c9c6cac2c47e1a",
  "eval/runner_agent.py": "5d85826014738e4a4f5f48ba119e09974106d9ec0953b8c6f63e99fa0b05067b",
  "eval/runner_agent_ext.py": "4d0b632e0ba47aaf20bda4f7ef66d0dc79b82d46c0f814dfe2f77184d2dba082",
  "eval/runner_stage2.py": "2cc939af3e59f8bc13a71eeedbd6cebeaa81578c20771cf8e8820a01f8c03132",
  "eval/runner_stage2_ext.py": "12bd8cdbbf6420a30b7508992fb6f8d4df5c3e11d8df2df02a47982c0db0aa12",
  "eval/runner_stage2_prompts.py": "5161daa8e82f545a312e96760419b4f1346d00e0920dc0212ffc6ee6275884a4",
  "eval/safety_table.py": "02a4d5dc627280577e35270463df8794a9dd02cfad8e38dbfa979db1d1daebf3",
  "eval/score_parsers.py": "cb4a7647551d2151a3a8d932ee7893ab5b2d730266404d38c23d6c4ae1b695b3",
  "eval/show_trace.py": "f88e0fdf172c66e45731c1b43b49318733f6ee017a960ecedc207f8f75af8160",
  "eval/stage2_metrics.py": "1d51e75f0dc612e5bd10da20dea4bb42e3167a18c2f8d877abe963edf1ebc84f",
  "eval/trajectory_labels.py": "17079b5cca09e838dfe95781058c2dcc6b13362e38e983d2b9e3233a56a48a22",
  "eval/with_sandbox.py": "7f57d2bd413adf22fccb1a64e1ea9caa6f3bc986e4be0c046ef1487999a6b0df",
  "requirements.txt": "299c954f0a528c78e9d5ef52238407bf9642c8d26689fcc26a03079cd945ad5d",
  "src/igt/__init__.py": "cfd91b71c91d591657ad2446817b9f04c3e3a2a8f6b50ba5ff5c6ee8d4fe2dec",
  "src/igt/agents/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
  "src/igt/agents/code_extract.py": "2bf4c28bd2a2d52724ffa0af028849d56a9d370eedfd60f8cdbd75c9fdeaac68",
  "src/igt/agents/code_generator.py": "6d997a58b03bd0a7d3cfff69d8fc6038119f419bb5287d9db53019ca4bd15c7f",
  "src/igt/agents/code_generator_v2.py": "87b1d8a1e282e083e5474c13e27d0474e338c63fb8e74df8cd0fb15f65d2e5f4",
  "src/igt/agents/intent_parser.py": "bf7e815d562338b27f2333939bf5d2fd5c4cef5a61d8350232989a1fd15977e6",
  "src/igt/agents/reask.py": "a30e82821ee4ecb5a582cd37b9e7813548559353bd8f40e1b6a9ca44eb841238",
  "src/igt/agents/repair_agent.py": "f37f5a3dad366d1f2c563aa183c23c13817a88f25bacaade1f4a1dd63a65ebff",
  "src/igt/agents/repair_agent_ext.py": "63eb91f81b3405080bd7eca27d8cdd36450455ce2c1fdc315d8ea769b70ea2cc",
  "src/igt/agents/repair_agent_v5.py": "71902b0af25c9f4a51174b1b78982f6fd659dc191adbc956d602bedae21d8b9a",
  "src/igt/agents/tool_parser.py": "9f8d7d75b0b5e1b0d86ed6b8d1e8d040dd951ee2ab966cf9a463eacaa61fbbd1",
  "src/igt/agents/tool_parser_ft.py": "f44c78b08eb4f1b238576241cce223aa0a90e2c2f395e8d4b4fcbfdd418170a6",
  "src/igt/baselines/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
  "src/igt/baselines/conventional_checks.py": "059fd6bc304c8243e99f3f8e8601f607bd5776fe0796ed68469faafc2154e132",
  "src/igt/baselines/llm_verifier.py": "64cef7c342befb24d494e5d636697bb71f0ef7d612ea033a6348802095e7e85c",
  "src/igt/config.py": "d884c6d150c3907fc5b6b976d684f544ac3c5435eb6d1a66daff25d54b54200e",
  "src/igt/contracts/__init__.py": "6f71bfcd9628b27674f28168aceab327edd2e6366772ab4f9c282af22b4e1159",
  "src/igt/contracts/normalize.py": "ecb9f07db2069127c69bb380893cfbdd465adb4622a0c8d02a9a3c3dbc598fe1",
  "src/igt/contracts/precheck.py": "6d10e297956bef3e8c34538a4da9f414112cd9a103e2be2f71e803e36f201f35",
  "src/igt/contracts/schema.py": "7981e85d79c7637cc424c7c2de14912799d43f1a19e663876253be0587d8c177",
  "src/igt/llm/__init__.py": "b28c1501ba0ffcaf09ad446403948b59f3264f8976d9206fd55921678095b6ec",
  "src/igt/llm/client.py": "29fb5d1be6290af2d462a934e7c3c96441c0f51e6afcc5754a9ed6d4d153bf94",
  "src/igt/llm/hf.py": "44128b0e596dea033ab3695dbf2e9f2af18d528d1f1150d8b96a8c274d8fadc7",
  "src/igt/llm/vllm_backend.py": "f3c0801756265c3fff93c18c8750101cb691f7ebb78ef748c9a53d7bd74e6c31",
  "src/igt/orchestrator/__init__.py": "4b871b816a82f19dc177ec79ba9edfa1b4dc15009edfc7d43964c383887178e6",
  "src/igt/orchestrator/limits.py": "3b23108054213c2750ec9fb0b951e8619d47121e18402a6883369653ae926319",
  "src/igt/orchestrator/memory.py": "8bf289afd7ad93f66a3eecb0c00ac0ac3b41f8fcf0bb099e838d95e996665459",
  "src/igt/orchestrator/state_machine.py": "502301f7f3d3b9b25c471e52cc70ed20f9693e4470d36a902c3392ecfb13f3d0",
  "src/igt/paths.py": "60f82a25b0a3648ebe2cd5f63f3435d2e6719a4c51ae0a3fd4e9a8e566f80bac",
  "src/igt/pipeline/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
  "src/igt/pipeline/failure_report.py": "845254415e0e3ccc34b70e865ebcb2f6b5b8df78038f2609ac5c473868a81348",
  "src/igt/pipeline/trace_view.py": "9d230b51d7621319f4c361eeea014505c401c2674607d715f39a7f9d9b5a708c",
  "src/igt/sandbox/__init__.py": "ec88bc3e6755ea281e6886cc37f747f475e4930f4096250bc133da624e05da25",
  "src/igt/sandbox/ast_guard.py": "fea71e9166df4b1f52f8bafc93ed147684991bf9d96739378ecf762bef90ece6",
  "src/igt/sandbox/docker_executor.py": "d25b83f7d2ca3aec266e9f203a53b06bf6052f3ee9391ecbafb0245d95a0dc97",
  "src/igt/sandbox/environment.py": "31b3eba1bb31276aa06695d93684e14dde71f59d64e39971ec07d152e13c11a6",
  "src/igt/sandbox/executor.py": "45272e1a360b020622a334f25997e3cbe59d668d692002cc1facc15ba324254b",
  "src/igt/tools/__init__.py": "0454b723f5f1dd186fc253def822cf0bcc0bdf9403573baa0dc1355ac8d38a4f",
  "src/igt/tools/data_tools.py": "6a813c2cbf968d7d363eaca3214aeaf7959772059634f392ed837f34d003f13d",
  "src/igt/tools/faults.py": "9c817ab9516551bc3be511549cc39910838bcaef64cc38fd5c65421cc661f054",
  "src/igt/tools/registry.py": "4a359df2ebd25e00ef34f38e7ab253dd4a20ef3100d4b0a026a1f183f46cedef",
  "src/igt/types.py": "099de8673b3927db05b97e888cb3c275a1f83ce1287332634c96dbabf88147b8",
  "src/igt/verifier/__init__.py": "8c8e2d7f849852f49cb009975936ad884601c582e389f4ea30601ef7a90d3086",
  "src/igt/verifier/alignment.py": "06281e0a013c2f030624edfecaed87891f89608e397768dfd071b39b83ca2e57",
  "src/igt/verifier/conditions.py": "8c48a413c93a72c62e17d9c90170db5af7925ab94658a31b0fb2c8c8cf2813f6",
  "src/igt/verifier/functions.py": "72398f12a3e395f9750922d46a936c88536ea7867ff4757b2c4d2e412c2961d3",
  "src/igt/verifier/intent_verifier.py": "c270014d9c1f07cb03ddcf53f319619afca0434afbbc465f3664a08485a93178",
  "src/igt/verifier/predicates.py": "82c5e4e75059cce09fd4947093b441a9de233996c534cabbb6e3745f41d4f5ba",
  "src/igt/verifier/values.py": "36bb324ccf9751363c2380784b0768ae9a52b439c87b5222649ee1494c0c97ad",
  "kaggle_outputs/parsing/7b_p1-v3/parses.jsonl": "451066e287428aba06d9a0db342fa126163df026c98af675cb24348191e8b1a7",
  "kaggle_outputs/parsing/ft_1_5b/parses.jsonl": "0aa69b6ba5622bd953145c355ae2ca9732110d9986e5b2e31d81a8bb31a3d4fd"
}
missing = [p for p in FROZEN_SOURCES if not Path(p).is_file()]
assert not missing, f"Missing required repository inputs: {missing}"
changed = [p for p, h in FROZEN_SOURCES.items() if hashlib.sha256(Path(p).read_bytes()).hexdigest() != h]
assert not changed, f"Frozen inputs/source differ; stop before test inference: {changed}"
print("Verified", len(FROZEN_SOURCES), "source/input files including both parser outputs.")

Verified 218 source/input files including both parser outputs.


In [4]:
# Install pinned data libraries into an isolated target, leaving the CUDA stack unchanged.
PACKAGES = Path(WORK) / "stage2_test_evalpkgs"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--target", str(PACKAGES),
                "-r", "requirements.txt"], check=True)
# Use the exact bitsandbytes version recorded in development.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes==0.50.2", "peft==0.21.2"], check=True)
ENV = dict(os.environ, PYTHONPATH=str(PACKAGES), TOKENIZERS_PARALLELISM="false")
probe = """import json, importlib.metadata, hashlib, pandas, numpy, torch
from igt.agents import code_generator_v2, repair_agent
v = {"pandas": pandas.__version__, "numpy": numpy.__version__}
for name in ["torch", "transformers", "bitsandbytes", "peft"]:
    try: v[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: v[name] = None
v.update(cuda_available=torch.cuda.is_available(), gpu_count=torch.cuda.device_count(),
         gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
         generator_hash=code_generator_v2.PROMPT_HASH,
         agent_hash=hashlib.sha256(repair_agent.FULL_SYSTEM.encode()).hexdigest()[:12])
print(json.dumps(v))
"""
# The source tree is added explicitly for this parent probe; model subprocesses add it themselves.
probe_env = dict(ENV, PYTHONPATH=str(REPO / "src") + os.pathsep + str(PACKAGES))
p = subprocess.run([sys.executable, "-c", probe], env=probe_env, capture_output=True, text=True)
assert p.returncode == 0, p.stderr
VERSIONS = json.loads(p.stdout.strip().splitlines()[-1])
assert VERSIONS["pandas"] == "2.3.3" and VERSIONS["numpy"] == "2.1.3"
mismatch = {k: (VERSIONS.get(k), v) for k, v in EXPECTED_INFERENCE.items() if VERSIONS.get(k) != v}
assert not mismatch, f"Inference environment differs from frozen development versions (actual, expected): {mismatch}"
assert VERSIONS["cuda_available"] and VERSIONS["gpu_count"] == 2, "Select Kaggle GPU T4 x2, as used in development"
assert all("T4" in gpu for gpu in VERSIONS["gpus"]), VERSIONS["gpus"]
assert VERSIONS["generator_hash"] == EXPECTED_GENERATOR_HASH
assert VERSIONS["agent_hash"] == EXPECTED_AGENT_HASH
ENV_TAG = hashlib.sha256(json.dumps(VERSIONS, sort_keys=True).encode()).hexdigest()[:12]
OUT = Path(WORK) / "results" / "stage2_test_frozen"
CACHE = Path(WORK) / "stage2_test_cache" / ENV_TAG
BUNDLE = Path(WORK) / "stage2_test_bundle.zip"
print("Verified parent and inference versions:", json.dumps(VERSIONS, indent=2))
print("Each model run also independently probes the actual child sandbox before model loading.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.33.3 requires PyYAML>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
d

Verified parent and inference versions: {
  "pandas": "2.3.3",
  "numpy": "2.1.3",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "bitsandbytes": "0.50.2",
  "peft": "0.21.2",
  "cuda_available": true,
  "gpu_count": 2,
  "gpus": [
    "Tesla T4",
    "Tesla T4"
  ],
  "generator_hash": "ed85f90bc61a",
  "agent_hash": "af55b596106e"
}
Each model run also independently probes the actual child sandbox before model loading.


In [5]:
# Check exact test coverage and freeze the six-arm plan before starting any model.
# This reads the evaluator's task inventory, not reference solutions into model prompts.
inventory = """import json, yaml
from pathlib import Path
rows=[yaml.safe_load(p.read_text()) for p in sorted(Path('benchmark/tasks').glob('*/*.yaml'))]
print(json.dumps([r['id'] for r in rows if r['split']=='test']))
"""
p = subprocess.run([sys.executable, "-c", inventory], env=ENV, capture_output=True, text=True)
assert p.returncode == 0, p.stderr
TEST_IDS = json.loads(p.stdout)
assert len(TEST_IDS) == 100 and len(set(TEST_IDS)) == 100
for filename in [B3_PARSES, FT_PARSES]:
    rows = [json.loads(line) for line in Path(filename).read_text().splitlines() if line.strip()]
    ids = [r["task_id"] for r in rows if r["split"] == "test"]
    assert len(ids) == 100 and set(ids) == set(TEST_IDS), f"Incomplete or duplicate test parses: {filename}"
FROZEN_PLAN = dict(
    plan_version="stage2-test-b2v2-a1v4-sb1", stage=STAGE, commit=COMMIT, backend=BACKEND,
    systems=ALL_SYSTEMS, test_ids=TEST_IDS, seeds=SEEDS, temperature=TEMPERATURE,
    generator_model=GENERATOR_MODEL, generator_4bit=GENERATOR_4BIT,
    generator_prompt=GENERATOR_PROMPT, generator_hash=EXPECTED_GENERATOR_HASH,
    agent_model=AGENT_MODEL, agent_4bit=AGENT_4BIT, agent_prompt=AGENT_PROMPT,
    agent_hash=EXPECTED_AGENT_HASH, max_attempts=MAX_ATTEMPTS,
    max_tool_calls=MAX_TOOL_CALLS, token_budget=TOKEN_BUDGET,
    generator_max_new_tokens=GENERATOR_MAX_NEW_TOKENS, dtype=DTYPE,
    b3_parses=B3_PARSES, ft_parses=FT_PARSES, ft_tag=FT_TAG,
    versions=VERSIONS, environment_tag=ENV_TAG,
    sources=FROZEN_SOURCES,
)
print("Plan: 6 systems x 100 test tasks x 1 seed = 600 records.")

Plan: 6 systems x 100 test tasks x 1 seed = 600 records.


In [6]:
# Helpers: deterministic names, commands, safe resume and complete/partial bundle reporting.
def system_name(preset):
    label = preset + "__gen_3b"
    if preset in ("proposed", "no_contract_to_generator"):
        label += "__" + FT_TAG
    gen = "b2v2" if preset in ("proposed", "gold_contract") else "b1v1"
    return label + "__" + gen + "__a1v4__sb1"


def build_command(preset):
    # The versioned runner appends prompt/sandbox tags; supply only the base name here.
    label = preset + "__gen_3b"
    if preset in ("proposed", "no_contract_to_generator"):
        label += "__" + FT_TAG
    cmd = [sys.executable, "eval/runner_stage2_prompts.py", "--preset", preset, "--system", label,
           "--splits", "test", "--backend", BACKEND,
           "--generator-prompt", GENERATOR_PROMPT, "--agent-prompt", AGENT_PROMPT,
           "--generator-model", GENERATOR_MODEL, "--agent-model", AGENT_MODEL,
           "--seeds", *map(str, SEEDS), "--temperature", str(TEMPERATURE),
           "--max-attempts", str(MAX_ATTEMPTS), "--max-tool-calls", str(MAX_TOOL_CALLS),
           "--token-budget", str(TOKEN_BUDGET), "--generator-max-new-tokens", str(GENERATOR_MAX_NEW_TOKENS),
           "--dtype", DTYPE, "--sandbox-packages", str(PACKAGES),
           "--cache-dir", str(CACHE), "--out", str(OUT)]
    if GENERATOR_4BIT: cmd.append("--generator-4bit")
    if AGENT_4BIT: cmd.append("--agent-4bit")
    if preset == "baseline3_repair": cmd += ["--parses", B3_PARSES]
    if preset in ("proposed", "no_contract_to_generator"): cmd += ["--parses", FT_PARSES]
    return cmd


def restore_bundle(filename):
    # Only accept our exact frozen plan and regular files within the designated output prefix.
    prefix = PurePosixPath("results/stage2_test_frozen")
    with zipfile.ZipFile(filename) as archive:
        name = str(prefix / "frozen_plan.json")
        assert json.loads(archive.read(name)) == FROZEN_PLAN, "Resume bundle belongs to a different experiment/environment"
        targets = []
        for entry in archive.infolist():
            path = PurePosixPath(entry.filename)
            assert not path.is_absolute() and ".." not in path.parts, "Unsafe archive path"
            assert path.is_relative_to(prefix), "Unexpected archive prefix"
            assert not stat.S_ISLNK(entry.external_attr >> 16), "Archive links are not supported"
            if not entry.is_dir():
                target = OUT.joinpath(*path.relative_to(prefix).parts)
                assert target.resolve().is_relative_to(OUT.resolve()), "Unsafe resume destination"
                data = archive.read(entry)
                assert not target.exists() or target.read_bytes() == data, f"Conflicting resume file: {target}"
                targets.append((target, data))
        # Validate the entire archive before writing anything.
        for target, data in targets:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)


def coverage(preset):
    path = OUT / system_name(preset) / "runs.jsonl"
    if not path.exists(): return dict(records=0, complete=False)
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    expected = {(tid, seed) for tid in TEST_IDS for seed in SEEDS}
    actual = [(r["task_id"], r["seed"]) for r in rows]
    assert len(actual) == len(set(actual)), f"Duplicate runs in {path}"
    assert set(actual) <= expected and all(r["split"] == "test" for r in rows), f"Unexpected tasks in {path}"
    for row in rows:
        assert row["system"] == system_name(preset) and row["preset"] == preset, f"Wrong system in {path}"
        assert row["agent_prompt_version"] == AGENT_PROMPT, f"Wrong repair prompt in {path}"
        assert row["generator_model"] == GENERATOR_MODEL and row["agent_model"] == AGENT_MODEL
    info_path = path.parent / "run_info.json"
    prompt_path = path.parent / "prompt_run_info.json"
    if info_path.exists():
        info = json.loads(info_path.read_text())
        config = info["config"]
        expected_gen = "b2-v2" if preset in ("proposed", "gold_contract") else "b1-v1"
        assert config["generator_prompt"]["version"] == expected_gen
        assert config["generator_prompt"]["hash"] == (EXPECTED_GENERATOR_HASH if expected_gen=="b2-v2" else "ebbe8bceae71")
        assert config["agent_prompt_version"] == AGENT_PROMPT and config["agent_prompt_hash"] == EXPECTED_AGENT_HASH
        assert config["backend"] == BACKEND and info["valid_result"] == (BACKEND=="hf")
        for name in ["max_attempts", "max_tool_calls", "token_budget"]:
            assert config[name] == FROZEN_PLAN[name], f"Changed budget: {name}"
    return dict(records=len(rows), complete=set(actual)==expected and info_path.is_file() and prompt_path.is_file())


def export_bundle(status):
    OUT.mkdir(parents=True, exist_ok=True)
    # Metrics can summarize partial systems; completeness below determines whether the experiment is final.
    if any(OUT.glob("*/runs.jsonl")):
        result = subprocess.run([sys.executable, "eval/stage2_metrics.py", str(OUT), "test"],
                                env=ENV, capture_output=True, text=True)
        (OUT / "metrics_test.txt").write_text(result.stdout + "\n" + result.stderr)
        assert result.returncode == 0, result.stderr
    counts = {preset: coverage(preset) for preset in ALL_SYSTEMS}
    complete = all(v["complete"] for v in counts.values())
    info = dict(status="complete" if complete else status, experiment_complete=complete,
                completed_records=sum(v["records"] for v in counts.values()), expected_records=len(ALL_SYSTEMS)*len(TEST_IDS)*len(SEEDS),
                coverage=counts, selected_this_session=SYSTEMS, commit=COMMIT,
                stage="test", backend=BACKEND, versions=VERSIONS)
    (OUT / "bundle_info.json").write_text(json.dumps(info, indent=2))
    for file in OUT.glob("*/run_info.json"):
        (file.parent / "git_commit.txt").write_text(COMMIT)
    temporary = BUNDLE.with_name(BUNDLE.stem + ".tmp.zip")
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUT.rglob("*")):
            if path.is_file():
                archive.write(path, str(PurePosixPath("results/stage2_test_frozen") / path.relative_to(OUT).as_posix()))
    temporary.replace(BUNDLE)
    print("Checkpoint:", info["completed_records"], "/ 600 records; complete:", complete, "->", BUNDLE, flush=True)
    return info

In [7]:
# Attach the previous stage2_test_bundle.zip as a Kaggle input first.
def find_resume_bundle(input_root, work):
    candidates = sorted(input_root.rglob("stage2_test_bundle.zip"))
    if candidates:
        assert len(candidates) == 1, (
            "Several checkpoint ZIPs were found; set RESUME_BUNDLE to the exact path: "
            f"{[str(path) for path in candidates]}"
        )
        return str(candidates[0])
    # Dataset upload can unpack a ZIP automatically. Repackage its files, then apply
    # the same strict frozen-plan and archive checks used for an intact ZIP.
    plans = sorted(input_root.rglob("frozen_plan.json"))
    assert len(plans) == 1, (
        "Attach the previous output ZIP as an input. Could not identify one checkpoint: "
        f"{[str(path) for path in plans]}"
    )
    extracted = plans[0].parent
    rebuilt = Path(work) / "stage2_resume_input.zip"
    with zipfile.ZipFile(rebuilt, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(extracted.rglob("*")):
            assert not path.is_symlink(), "Checkpoint input links are not supported"
            if path.is_file():
                archive.write(path, "results/stage2_test_frozen/" + path.relative_to(extracted).as_posix())
    print("Kaggle unpacked the input ZIP; recovered checkpoint from:", extracted)
    return str(rebuilt)


if RESUME_BUNDLE == "auto":
    RESUME_BUNDLE = find_resume_bundle(Path("/kaggle/input"), WORK)
assert RESUME_BUNDLE and Path(RESUME_BUNDLE).is_file(), "A previous checkpoint ZIP is required"
print("Resume checkpoint:", RESUME_BUNDLE)

# Restore optional prior output. Refuse to mix runs from another plan, source or environment.
if RESUME_BUNDLE:
    if OUT.exists() and any(OUT.iterdir()):
        print("Retaining current session outputs; not restoring an older checkpoint over them.")
    else:
        restore_bundle(RESUME_BUNDLE)
if OUT.exists() and any(OUT.iterdir()):
    plan_file = OUT / "frozen_plan.json"
    assert plan_file.is_file() and json.loads(plan_file.read_text()) == FROZEN_PLAN, "Existing outputs do not match the frozen plan"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "frozen_plan.json").write_text(json.dumps(FROZEN_PLAN, indent=2))
(OUT / "source_manifest.json").write_text(json.dumps(FROZEN_SOURCES, indent=2))
LOGS = OUT / "logs"
LOGS.mkdir(exist_ok=True)
for preset in ALL_SYSTEMS:
    print(preset, coverage(preset))

Resume checkpoint: /kaggle/input/notebooks/srinivasu2005/stage-2-eval/stage2_test_bundle.zip
baseline1 {'records': 100, 'complete': True}
baseline2_repair {'records': 100, 'complete': True}
baseline3_repair {'records': 100, 'complete': True}
gold_contract {'records': 100, 'complete': True}
proposed {'records': 100, 'complete': True}
no_contract_to_generator {'records': 62, 'complete': False}


In [8]:
GPU_SPLIT_BOOTSTRAP = r'''"""Process-local GPU placement; the frozen repository is never edited."""
import json
import os
import runpy
import sys
from pathlib import Path


def install_placement(model_class, placements, record):
    original = model_class.from_pretrained

    def load(model_id, *args, **kwargs):
        if model_id not in placements:
            raise RuntimeError(f"Unexpected model in recovery process: {model_id}")
        device = placements[model_id]
        kwargs["device_map"] = {"": device}
        model = original(model_id, *args, **kwargs)
        actual = {str(parameter.device) for parameter in model.parameters()}
        if actual != {f"cuda:{device}"}:
            raise RuntimeError(f"Placement check failed for {model_id}: {actual}")
        # Single-device loading may omit this optional metadata.
        # The parameter-device check above establishes actual placement.
        reported_map = getattr(model, "hf_device_map", None) or {}
        record(dict(model=model_id, requested_device=f"cuda:{device}",
                    parameter_devices=sorted(actual),
                    hf_device_map={str(k): str(v) for k, v in reported_map.items()}))
        return model

    model_class.from_pretrained = staticmethod(load)


def main():
    import torch
    from transformers import AutoModelForCausalLM

    assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Requires GPU T4 x2"
    provenance = Path(os.environ["STAGE2_PLACEMENT_RECORD"])
    metadata = json.loads(provenance.read_text())

    def record(item):
        metadata["model_loads"].append(item)
        temporary = provenance.with_suffix(".tmp")
        temporary.write_text(json.dumps(metadata, indent=2))
        temporary.replace(provenance)
        print("Verified GPU placement:", json.dumps(item), flush=True)

    install_placement(AutoModelForCausalLM, metadata["model_gpu_ordinals"], record)
    runner = Path(sys.argv[1]).resolve()
    # Match direct execution of the original runner (imports, argv, and source path).
    sys.path.insert(0, str(runner.parent))
    sys.argv = [str(runner), *sys.argv[2:]]
    runpy.run_path(str(runner), run_name="__main__")


if __name__ == "__main__":
    main()
'''

# Record the runtime intervention separately from the unchanged frozen method.
# Explicit placement is supported by Transformers:
# https://huggingface.co/docs/transformers/main_classes/model
assert SYSTEMS == ["no_contract_to_generator"], "This notebook resumes only the unfinished arm"
assert GENERATOR_MODEL != AGENT_MODEL
RECOVERY_VERSION = "gpu-split-v2"
checkpoint_counts = {preset: coverage(preset) for preset in ALL_SYSTEMS}
assert all(checkpoint_counts[preset]["complete"] for preset in ALL_SYSTEMS if preset not in SYSTEMS), (
    "The other five systems must already be complete in this checkpoint. "
    "Attach the full previous output bundle."
)
PRESERVED_FILES = {
    str(path.relative_to(OUT)): path.read_bytes()
    for path in OUT.glob("*/*")
    if path.name in ("runs.jsonl", "traces.jsonl", "run_info.json", "prompt_run_info.json")
}
before_keys = {}
for path in OUT.glob("*/runs.jsonl"):
    before_keys[path.parent.name] = [
        [row["task_id"], row["seed"]]
        for row in (json.loads(line) for line in path.read_text().splitlines() if line.strip())
    ]
completed_before = sum(len(keys) for keys in before_keys.values())
assert completed_before > 0, "No saved runs were restored; do not start a fresh test here"
BOOTSTRAP_PATH = Path(WORK) / "stage2_gpu_split_bootstrap.py"
BOOTSTRAP_PATH.write_text(GPU_SPLIT_BOOTSTRAP)
INTERVENTIONS = OUT / "runtime_interventions"
INTERVENTIONS.mkdir(exist_ok=True)
PLACEMENT_RECORD = INTERVENTIONS / f"gpu_split_resume_{time.time_ns()}.json"
PLACEMENT_RECORD.write_text(json.dumps(dict(
    intervention=RECOVERY_VERSION, reason="CUDA OOM in repair generation on GPU 0",
    scope="new model calls in this resume session only; previous records retained",
    previous_device_map="auto independently for each model",
    model_gpu_ordinals={AGENT_MODEL: 0, GENERATOR_MODEL: 1},
    source_commit=COMMIT, source_files_edited=False,
    bootstrap_sha256=hashlib.sha256(GPU_SPLIT_BOOTSTRAP.encode()).hexdigest(),
    bootstrap_source=GPU_SPLIT_BOOTSTRAP,
    versions=VERSIONS, checkpoint_sha256=hashlib.sha256(Path(RESUME_BUNDLE).read_bytes()).hexdigest(),
    completed_before=completed_before, existing_task_seed_keys=before_keys,
    selected_systems=SYSTEMS, model_loads=[],
    limitation="GPU placement differs from the earlier run; bitwise output equivalence is not established. "
               "Prompts, model IDs, precision, data, decoding and budgets retain the frozen settings."
), indent=2))
ENV = dict(ENV, STAGE2_PLACEMENT_RECORD=str(PLACEMENT_RECORD))
# Preserve the base command even if this setup cell is run repeatedly.
original_build_command = getattr(build_command, "_stage2_base_command", build_command)


def build_command(preset, _base=original_build_command, _bootstrap=BOOTSTRAP_PATH):
    command = _base(preset)
    return [command[0], str(_bootstrap), *command[1:]]


build_command._stage2_base_command = original_build_command


def verify_preserved_records():
    for relative, original in PRESERVED_FILES.items():
        current = (OUT / relative).read_bytes()
        if relative.endswith(".jsonl"):
            assert current.startswith(original), f"Existing records were changed: {relative}"
        else:
            assert current == original, f"Completed metadata was changed: {relative}"


print("Recovery revision:", RECOVERY_VERSION)
print("Preserved completed records:", completed_before)
print("Pending tasks:", len(TEST_IDS) * len(SEEDS) - coverage("no_contract_to_generator")["records"])
print("Placement: repair 7B four-bit -> GPU 0; generator 3B FP16 -> GPU 1")
print("Runtime intervention record:", PLACEMENT_RECORD)


Recovery revision: gpu-split-v2
Preserved completed records: 562
Pending tasks: 38
Placement: repair 7B four-bit -> GPU 0; generator 3B FP16 -> GPU 1
Runtime intervention record: /kaggle/working/results/stage2_test_frozen/runtime_interventions/gpu_split_resume_1794550012018470858.json


In [9]:
# This cell can also be rerun after the final download/summary cell.
os.chdir(REPO)
failure = None
try:
    # Keep a safe, downloadable checkpoint before any new GPU process starts.
    verify_preserved_records()
    export_bundle("partial")
    for preset in SYSTEMS:
        if coverage(preset)["complete"]:
            print("Already complete; skipping", preset)
            continue
        command = build_command(preset)
        log_path = LOGS / (system_name(preset) + ".log")
        started = time.time()
        print("Running", preset, flush=True)
        with log_path.open("a") as log:
            log.write("Command: " + json.dumps(command) + "\n")
            proc = subprocess.Popen(command, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, bufsize=1)
            try:
                for line in proc.stdout:
                    log.write(line)
                    log.flush()
                    if line.startswith(("[seed", "Verified sandbox:", "Verified GPU placement:", "runs in")):
                        print(line, end="", flush=True)
                code = proc.wait()
            except BaseException:
                proc.terminate()
                try: proc.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
                raise
            log.write(f"\nExit: {code}; minutes: {(time.time()-started)/60:.2f}\n")
        if code:
            print(log_path.read_text()[-7000:])
            raise RuntimeError(f"{preset} failed; inspect {log_path}. Download the partial bundle and resume after diagnosis.")
        assert coverage(preset)["complete"], f"{preset} did not produce all 100 test records"
        verify_preserved_records()
        export_bundle("partial")
        print("Completed", preset, "in", round((time.time()-started)/60, 1), "minutes")
except BaseException as error:
    failure = error
finally:
    try:
        verify_preserved_records()
    except Exception as error:
        print("Preservation check failed; retaining the last safe ZIP:", error)
        if failure is None:
            failure = error
    else:
        try:
            progress = export_bundle("failed" if failure is not None else "partial")
        except Exception as error:
            print("Checkpoint export failed; retaining the last safe ZIP:", error)
            if failure is None:
                failure = error
    from IPython.display import display, FileLink
    try:
        os.chdir(WORK)
        if BUNDLE.is_file():
            display(FileLink(BUNDLE.name))
    finally:
        os.chdir(REPO)
if failure is not None:
    raise failure


Checkpoint: 562 / 600 records; complete: False -> /kaggle/working/stage2_test_bundle.zip
Running no_contract_to_generator
Verified sandbox: {"executable": "/usr/bin/python3", "numpy": "2.1.3", "numpy_file": "/kaggle/working/stage2_test_evalpkgs/numpy/__init__.py", "package_root": "/kaggle/working/stage2_test_evalpkgs", "pandas": "2.3.3", "pandas_file": "/kaggle/working/stage2_test_evalpkgs/pandas/__init__.py", "python": "3.13.16", "version": "sb1"}
Verified GPU placement: {"model": "Qwen/Qwen2.5-7B-Instruct", "requested_device": "cuda:0", "parameter_devices": ["cuda:0"], "hf_device_map": {}}
Verified GPU placement: {"model": "Qwen/Qwen2.5-3B-Instruct", "requested_device": "cuda:1", "parameter_devices": ["cuda:1"], "hf_device_map": {}}
ne 88, in _run_code
  File "/kaggle/working/repo_stage2_test_frozen/eval/runner_stage2_prompts.py", line 82, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/repo_stage2_test_frozen/eval/runner_stage2_prompts.py", line 70, in main
    out = runne

/kaggle/working/stage2_test_bundle.zip

RuntimeError: no_contract_to_generator failed; inspect /kaggle/working/results/stage2_test_frozen/logs/no_contract_to_generator__gen_3b__ft_1_5b__b1v1__a1v4__sb1.log. Download the partial bundle and resume after diagnosis.

In [ ]:
# Inspect completeness before interpreting the scores as final test results.
progress = json.loads((OUT / "bundle_info.json").read_text())
print(json.dumps(progress, indent=2))
if (OUT / "metrics_test.txt").exists():
    print((OUT / "metrics_test.txt").read_text())
if progress["experiment_complete"]:
    print("COMPLETE: all six systems have all 100 test tasks. Freeze and download this bundle for analysis.")
else:
    remaining = [name for name, info in progress["coverage"].items() if not info["complete"]]
    print("PARTIAL, not final results. Remaining systems:", remaining)
    print("Download this checkpoint, attach it in the next session, and set RESUME_BUNDLE.")
print("Download", BUNDLE)
from IPython.display import display, FileLink
os.chdir(WORK)
display(FileLink(BUNDLE.name))

## After resuming

Keep the previous checkpoint and download the newly exported **stage2_test_bundle.zip**, plus this executed notebook. Confirm **600 records**, **100 per system**, and `experiment_complete: true`. The resumed bundle includes the earlier OOM log and a `runtime_interventions/` record documenting the change in GPU placement. Report that intervention with the final experiment; do not describe all records as having identical device placement. If another error occurs, the failure handler exports the partial checkpoint again. Do not change prompts, quantization or budgets to work around a failure.
